In [39]:
import os
import sys
import umap

import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt

from core.data.utils import read_all_datasets, read_all_metadata
from core.data.processing.normalization import robust_zscore_normalization_per_dataset
from core.stats.tests import test_two_groups
from core.utils.genes import entrez_id_to_gene_symbol

In [40]:
libraries = gp.get_library_name(organism='Human')

In [41]:
DATA_DIR = '../../db'
ENRICHR_LIBRARIES = ['WikiPathway_2023_Human', 'KEGG_2021_Human', 'GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'GO_Cellular_Component_2023', 'Reactome_2022']

In [42]:
dataset, dataset_label = read_all_datasets(DATA_DIR, dropna=False)

In [43]:
dataset = entrez_id_to_gene_symbol(dataset)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
4978 input query terms found no hit:	['100127886', '100127940', '100127972', '100127974', '100128046', '100128175', '100128185', '1001281


In [44]:
metadata = read_all_metadata(DATA_DIR)

In [45]:
metadata['inflammation'] = np.where(pd.isna(metadata['inflammation']) & (metadata['disease'] == 'UC'), 'Inflamed', metadata['inflammation'])
metadata['inflammation'] = np.where(pd.isna(metadata['inflammation']) & (metadata['disease'] == 'CD'), 'Inflamed', metadata['inflammation'])
metadata['inflammation'] = np.where(pd.isna(metadata['inflammation']) & (metadata['disease'] == 'Ctrl'), 'Uninflamed', metadata['inflammation'])

In [46]:
y = metadata[(((metadata.disease.isin(['UC', 'CD'])) & (metadata.inflammation == 'Inflamed')) | ((metadata.disease == 'Ctrl') & (metadata.inflammation == 'Uninflamed'))) & metadata.time_of_biopsy.isin([None, 'W0', 'Before', np.nan])].disease
dataset_labels = dataset_label.loc[y.index]
X = dataset.loc[y.index]

In [ ]:
"""
Gene expression differential analysis with pairwise comparisons,
multiple testing correction, and cross-dataset aggregation.
"""
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations


def pairwise_tests(X: pd.DataFrame, y: pd.Series, pairs: list = None) -> pd.DataFrame:
    """
    Pairwise Mann-Whitney U tests between groups for each gene.
    
    Args:
        X: samples x genes expression matrix
        y: group labels (e.g., 'UC', 'CD', 'Ctrl')
        pairs: specific pairs to test, or None for all combinations
    
    Returns:
        DataFrame with p-values and effect directions (genes x pairs)
        Direction: +1 if first group > second, -1 if first < second
    """
    if pairs is None:
        pairs = list(combinations(sorted(y.unique()), 2))
    
    def safe_mannwhitney(a, b):
        """Return NaN for degenerate cases (constant, all-NaN, <2 samples)."""
        a, b = a.dropna(), b.dropna()
        if len(a) < 2 or len(b) < 2:
            return np.nan
        if a.nunique() == 1 and b.nunique() == 1 and a.iloc[0] == b.iloc[0]:
            return 1.0  # identical constant distributions
        try:
            return stats.mannwhitneyu(a, b, alternative='two-sided')[1]
        except ValueError:
            return np.nan
    
    results = {}
    for g1, g2 in pairs:
        x1, x2 = X.loc[y == g1], X.loc[y == g2]
        pvals = [safe_mannwhitney(x1[g], x2[g]) for g in X.columns]
        # Direction: sign of median difference (g1 - g2)
        directions = np.sign(x1.median() - x2.median()).values
        
        results[f'{g1}_vs_{g2}'] = pvals
        results[f'{g1}_vs_{g2}_dir'] = directions
    
    return pd.DataFrame(results, index=X.columns)


def correct_and_flag(pvals: pd.DataFrame, alpha: float = 0.05, 
                     method: str = 'fdr_bh') -> pd.DataFrame:
    """
    Apply FDR correction across all tests, add significance flags.
    
    Returns DataFrame with: raw p-values, adjusted p-values (*_adj), 
    boolean significance flags (*_sig), and direction (*_dir).
    """
    if pvals.empty:
        raise ValueError("Empty p-value DataFrame")
    
    # Separate p-value and direction columns
    pval_cols = [c for c in pvals.columns if not c.endswith('_dir')]
    dir_cols = [c for c in pvals.columns if c.endswith('_dir')]
    
    # Handle NaNs: correct only valid p-values
    pval_data = pvals[pval_cols]
    flat = pval_data.values.flatten()
    valid_mask = ~np.isnan(flat)
    
    adj = np.full_like(flat, np.nan)
    if valid_mask.any():
        _, adj[valid_mask], _, _ = multipletests(flat[valid_mask], method=method)
    
    adj_df = pd.DataFrame(adj.reshape(pval_data.shape), 
                          index=pvals.index, columns=pval_cols)
    
    # Build result with columns grouped by comparison
    result = pd.DataFrame(index=pvals.index)
    for col in pval_cols:
        dir_col = f'{col}_dir'
        result[col] = pvals[col]
        result[f'{col}_adj'] = adj_df[col]
        result[f'{col}_sig'] = adj_df[col] < alpha
        if dir_col in pvals.columns:
            result[dir_col] = pvals[dir_col]
    
    return result


def analyze_by_dataset(X: pd.DataFrame, y: pd.Series, dataset_labels: pd.Series,
                       pairs: list = None, alpha: float = 0.05,
                       min_per_group: int = 3) -> dict:
    """
    Run pairwise tests separately for each dataset.
    
    Args:
        min_per_group: minimum samples per group to include dataset
    
    Returns:
        dict mapping dataset_name -> results DataFrame
    """
    if pairs is None:
        all_groups = sorted(y.unique())
        pairs = list(combinations(all_groups, 2))
    
    required_groups = set(g for pair in pairs for g in pair)
    results = {}
    skipped = []
    
    for ds in dataset_labels.unique():
        mask = dataset_labels == ds
        y_ds = y.loc[mask]
        
        # Check all required groups have enough samples
        counts = y_ds.value_counts()
        missing = required_groups - set(counts.index)
        too_few = [g for g in required_groups if counts.get(g, 0) < min_per_group]
        
        if missing or too_few:
            skipped.append((ds, missing, too_few))
            continue
        
        pvals = pairwise_tests(X.loc[mask], y_ds, pairs)
        results[ds] = correct_and_flag(pvals, alpha)
    
    if skipped:
        print(f"Skipped {len(skipped)} datasets with insufficient groups:")
        for ds, missing, too_few in skipped:
            if missing:
                print(f"  {ds}: missing groups {missing}")
            if too_few:
                print(f"  {ds}: <{min_per_group} samples in {too_few}")
    
    if not results:
        raise ValueError("No datasets have all required groups with sufficient samples")
    
    return results


def aggregate_significance(per_dataset: dict) -> pd.DataFrame:
    """
    Count significance with direction across datasets.
    
    Returns DataFrame with columns:
        - {pair}_n_up: significant with positive direction (g1 > g2)
        - {pair}_n_down: significant with negative direction (g1 < g2)
        - {pair}_n_sig: total significant (up + down)
        - n_datasets: total datasets analyzed
    """
    datasets = list(per_dataset.keys())
    first_ds = per_dataset[datasets[0]]
    
    # Get comparison names (without _adj, _sig, _dir suffixes)
    comparisons = [c for c in first_ds.columns if not c.endswith(('_adj', '_sig', '_dir'))]
    
    agg = pd.DataFrame(index=first_ds.index)
    
    for comp in comparisons:
        sig_col, dir_col = f'{comp}_sig', f'{comp}_dir'
        
        n_up = pd.Series(0, index=agg.index)
        n_down = pd.Series(0, index=agg.index)
        
        for ds in datasets:
            df = per_dataset[ds]
            is_sig = df[sig_col].fillna(False)
            direction = df[dir_col].fillna(0)
            
            n_up += (is_sig & (direction > 0)).astype(int)
            n_down += (is_sig & (direction < 0)).astype(int)
        
        agg[f'{comp}_n_up'] = n_up
        agg[f'{comp}_n_down'] = n_down
        agg[f'{comp}_n_sig'] = n_up + n_down
    
    agg['n_datasets'] = len(datasets)
    return agg


def meta_analysis(per_dataset: dict, method: str = 'fisher', alpha: float = 0.05) -> pd.DataFrame:
    """
    Combine p-values across datasets using Fisher's method.
    
    More powerful than vote counting - detects consistent weak effects.
    Returns combined p-values per comparison with FDR correction.
    """
    datasets = list(per_dataset.keys())
    pval_cols = [c for c in per_dataset[datasets[0]].columns 
                 if not c.endswith(('_adj', '_sig', '_dir'))]
    
    result = pd.DataFrame(index=per_dataset[datasets[0]].index)
    
    for col in pval_cols:
        # Stack p-values: genes × datasets
        pvals = np.column_stack([per_dataset[ds][col].values for ds in datasets])
        
        combined_p = []
        for row in pvals:
            valid = row[~np.isnan(row)]
            if len(valid) == 0:
                combined_p.append(np.nan)
            else:
                # Fisher's method: -2 * sum(log(p)) ~ chi2(2k)
                chi2_stat = -2 * np.sum(np.log(valid + 1e-300))
                combined_p.append(1 - stats.chi2.cdf(chi2_stat, df=2 * len(valid)))
        
        result[col] = combined_p
    
    # FDR correction across all combined p-values (handling NaN)
    flat = result.values.flatten()
    valid_mask = ~np.isnan(flat)
    adj = np.full_like(flat, np.nan)
    if valid_mask.any():
        _, adj[valid_mask], _, _ = multipletests(flat[valid_mask], method='fdr_bh')
    adj_matrix = adj.reshape(result.shape)
    
    for i, col in enumerate(result.columns):
        result[f'{col}_adj'] = adj_matrix[:, i]
        result[f'{col}_sig'] = adj_matrix[:, i] < alpha
    
    return result


def full_analysis(X: pd.DataFrame, y: pd.Series, dataset_labels: pd.Series,
                  alpha: float = 0.05, min_per_group: int = 3) -> tuple:
    """
    Complete analysis pipeline.
    
    Returns:
        (per_dataset_results, aggregated_significance)
    """
    per_ds = analyze_by_dataset(X, y, dataset_labels, alpha=alpha, 
                                 min_per_group=min_per_group)
    agg = aggregate_significance(per_ds)
    return per_ds, agg

In [82]:
per_ds, agg = full_analysis(X, y, dataset_labels)

meta = meta_analysis(per_ds)

Skipped 10 datasets with insufficient groups:
  GSE11223: missing groups {'CD'}
  GSE11223: <3 samples in ['CD']
  GSE23597: missing groups {'Ctrl', 'CD'}
  GSE23597: <3 samples in ['Ctrl', 'CD']
  GSE3629: missing groups {'Ctrl', 'CD'}
  GSE3629: <3 samples in ['Ctrl', 'CD']
  GSE92415: missing groups {'CD'}
  GSE92415: <3 samples in ['CD']
  GSE22619: missing groups {'CD'}
  GSE22619: <3 samples in ['CD']
  GSE9452: missing groups {'CD'}
  GSE9452: <3 samples in ['CD']
  GSE73661: missing groups {'CD'}
  GSE73661: <3 samples in ['CD']
  GSE87465: missing groups {'Ctrl', 'CD'}
  GSE87465: <3 samples in ['Ctrl', 'CD']
  GSE72780: missing groups {'UC'}
  GSE72780: <3 samples in ['UC']
  GSE52746: missing groups {'UC'}
  GSE52746: <3 samples in ['UC']


In [160]:
gene_list = agg[(agg['CD_vs_UC_n_sig'] > 1) & (agg['Ctrl_vs_UC_n_sig'] > 1)].sort_values(by=['CD_vs_UC_n_sig', 'Ctrl_vs_UC_n_sig'], ascending=False).index.tolist()

In [161]:
len(gene_list)

3103

In [ ]:
all_enrichment_results = pd.DataFrame()
for library in ENRICHR_LIBRARIES:
    enrichr_results = gp.enrichr(gene_list=gene_list, gene_sets=library, organism='Human', cutoff=0.05)
    enrichr_results.res2d['library'] = library
    all_enrichment_results = pd.concat([all_enrichment_results, enrichr_results.res2d])

In [172]:
def get_up(genes_str):
    """
    For each gene listed in genes_str, check if it is up or down-regulated in UC vs Ctrl.
    We want to return the proportion of genes up-regulated in UC.
    """
    genes = genes_str.split(';')

    # make sure genes are in the agg index
    genes = [g for g in genes if g in agg.index]
    if len(genes) == 0:
        return np.nan

    tmp = agg.loc[genes, 'Ctrl_vs_UC_n_down']
    return (tmp > 0).sum() / len(genes)

In [173]:
all_enrichment_results['UP_proportion'] = all_enrichment_results.apply(lambda row: get_up(row['Genes']), axis=1)

In [179]:
all_enrichment_results.sort_values(by='Combined Score', ascending=False).head(23)

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,library,UP_proportion
10,GO_Biological_Process_2023,Xenobiotic Glucuronidation (GO:0052697),6/6,1.389024e-05,6.009425e-03,0,0,101382.000000,1.133889e+06,UGT2B28;UGT2B4;UGT1A9;UGT1A8;UGT1A7;UGT1A6,GO_Biological_Process_2023,0.000000
29,Reactome_2022,Beta Oxidation Of octanoyl-CoA To hexanoyl-CoA...,5/5,8.965155e-05,5.071289e-03,0,0,84485.000000,7.873647e+05,HADHB;HADHA;ECHS1;ACADM;HADH,Reactome_2022,0.000000
28,Reactome_2022,Beta Oxidation Of hexanoyl-CoA To butanoyl-CoA...,5/5,8.965155e-05,5.071289e-03,0,0,84485.000000,7.873647e+05,HADHB;HADHA;ECHS1;HADH;ACADS,Reactome_2022,0.000000
63,WikiPathway_2023_Human,Vitamins A And D Action Mechanisms WP4342,3/3,3.731591e-03,4.501231e-02,0,0,50691.000000,2.834094e+05,RXRA;VDR;RARA,WikiPathway_2023_Human,0.000000
122,WikiPathway_2023_Human,PCSK9 Mediated LDL Receptor Degradation WP2846,2/2,2.406481e-02,1.489957e-01,0,0,33794.000000,1.259504e+05,PCSK9;LDLR,WikiPathway_2023_Human,1.000000
123,WikiPathway_2023_Human,Evolocumab Mechanism To Reduce LDL Cholesterol...,2/2,2.406481e-02,1.489957e-01,0,0,33794.000000,1.259504e+05,PCSK9;LDLR,WikiPathway_2023_Human,1.000000
2,GO_Biological_Process_2023,Glucuronate Metabolic Process (GO:0019585),13/15,2.275608e-09,3.609873e-06,0,0,35.539644,7.072751e+02,UGT2B10;UGT1A10;UGT2B15;UGT2B17;UGT2B28;UGT1A4...,GO_Biological_Process_2023,0.000000
1,GO_Biological_Process_2023,Cellular Glucuronidation (GO:0052695),13/15,2.275608e-09,3.609873e-06,0,0,35.539644,7.072751e+02,UGT2B10;UGT1A10;UGT2B15;UGT2B17;UGT2B28;UGT1A4...,GO_Biological_Process_2023,0.000000
2,WikiPathway_2023_Human,Mitochondrial Fatty Acid Oxidation Disorders W...,16/19,6.598575e-11,1.683205e-08,0,0,29.187345,6.841975e+02,PECR;CPT1A;ACADVL;SLC22A5;ECI1;ACSL3;ACSF2;HAD...,WikiPathway_2023_Human,0.062500
5,WikiPathway_2023_Human,Mitochondrial Long Chain Fatty Acid Beta Oxida...,14/17,1.946978e-09,2.302015e-07,0,0,25.522391,5.119023e+02,PECR;CPT1A;ACADVL;ECI1;ACSL3;ACSF2;HADHA;SCP2;...,WikiPathway_2023_Human,0.071429
